# DiffusionCIFAR: Improved Conditional DDPM Pipeline for CIFAR-10

This noteebook contains an improved training pipeline of conditional diffusion model on CIFAR-10.

## What has been improved
- Pipeline adapted for CIFAR-10 dataset
- **Class-conditional DDPM** with classifier-free guidance
- **UNet with residual-blocks and self-attention**
- **EMA** for a more stable sampling
- **AMP (mixed precision)** for faster training
- **Cosine LR scheduler + gradient clipping**
- Ready-to-use functions for **training, validation, and sampling**

In [ ]:
import os
import math
import copy
import random
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils

torch.backends.cudnn.benchmark = True

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
@dataclass
class CFG:
    # Data
    data_root: str = './data'
    image_size: int = 32
    num_classes: int = 10
    batch_size: int = 128
    num_workers: int = 4

    # Optimization
    epochs: int = 100
    lr: float = 2e-4
    weight_decay: float = 1e-4
    grad_clip: float = 1.0

    # Diffusion
    timesteps: int = 1000
    beta_start: float = 1e-4
    beta_end: float = 0.02

    # Model
    base_channels: int = 128
    channel_mults: tuple = (1, 2, 2, 4)
    attn_resolutions: tuple = (16,)
    dropout: float = 0.1

    # Training tricks
    label_drop_prob: float = 0.1   # classifier-free guidance training
    ema_decay: float = 0.9999
    use_amp: bool = True

    # Logging / sampling
    sample_every: int = 5
    num_sample_rows: int = 10
    guidance_scale: float = 3.0
    preview_guidance_scale: float = 1.5
    out_dir: str = './models/diffusion_cifar'

cfg = CFG()
os.makedirs(cfg.out_dir, exist_ok=True)
cfg

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


def linear_beta_schedule(timesteps, beta_start=1e-4, beta_end=0.02):
    return torch.linspace(beta_start, beta_end, timesteps)


def cosine_beta_schedule(timesteps, s=0.008):
    # Nichol & Dhariwal cosine schedule: usually learns/stabilizes better than linear
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return betas.clamp(1e-4, 0.999)


def extract(a, t, x_shape):
    # gather values for each batch item and reshape for broadcast
    b = t.shape[0]
    out = a.gather(-1, t)
    return out.reshape(b, *((1,) * (len(x_shape) - 1)))


def timestep_embedding(timesteps, dim, max_period=10000):
    half = dim // 2
    freqs = torch.exp(-math.log(max_period) * torch.arange(0, half, dtype=torch.float32, device=timesteps.device) / half)
    args = timesteps[:, None].float() * freqs[None]
    emb = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=-1)
    return emb

In [ ]:
train_tfms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

test_tfms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_ds = datasets.CIFAR10(root=cfg.data_root, train=True, download=True, transform=train_tfms)
test_ds = datasets.CIFAR10(root=cfg.data_root, train=False, download=True, transform=test_tfms)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=True,
    drop_last=True,
)

test_loader = DataLoader(
    test_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True,
)

classes = train_ds.classes
print('Classes:', classes)

# quick preview
x_preview, y_preview = next(iter(train_loader))
grid = utils.make_grid((x_preview[:16] * 0.5 + 0.5).clamp(0, 1), nrow=4)
plt.figure(figsize=(5, 5))
plt.imshow(np.transpose(grid.cpu().numpy(), (1, 2, 0)))
plt.axis('off')
plt.title('CIFAR-10 preview')
plt.show()

In [ ]:
class SiLU(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, emb_dim, dropout=0.0):
        super().__init__()
        self.in_layers = nn.Sequential(
            nn.GroupNorm(8, in_ch),
            SiLU(),
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
        )
        self.emb_layers = nn.Sequential(
            SiLU(),
            nn.Linear(emb_dim, out_ch),
        )
        self.out_layers = nn.Sequential(
            nn.GroupNorm(8, out_ch),
            SiLU(),
            nn.Dropout(dropout),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
        )
        self.skip = nn.Identity() if in_ch == out_ch else nn.Conv2d(in_ch, out_ch, 1)

    def forward(self, x, emb):
        h = self.in_layers(x)
        emb_out = self.emb_layers(emb)[:, :, None, None]
        h = h + emb_out
        h = self.out_layers(h)
        return h + self.skip(x)


class SelfAttention2d(nn.Module):
    def __init__(self, ch, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.norm = nn.GroupNorm(8, ch)
        self.qkv = nn.Conv1d(ch, ch * 3, 1)
        self.proj = nn.Conv1d(ch, ch, 1)

    def forward(self, x):
        b, c, h, w = x.shape
        x_in = x
        x = self.norm(x).view(b, c, h * w)
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=1)

        head_dim = c // self.num_heads
        q = q.view(b, self.num_heads, head_dim, h * w)
        k = k.view(b, self.num_heads, head_dim, h * w)
        v = v.view(b, self.num_heads, head_dim, h * w)

        scale = 1.0 / math.sqrt(head_dim)
        attn = torch.einsum('bncd,bnce->bnde', q, k) * scale
        attn = torch.softmax(attn, dim=-1)
        out = torch.einsum('bnde,bnce->bncd', attn, v)
        out = out.reshape(b, c, h * w)

        out = self.proj(out).view(b, c, h, w)
        return x_in + out


class Downsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.op = nn.Conv2d(ch, ch, kernel_size=3, stride=2, padding=1)

    def forward(self, x):
        return self.op(x)


class Upsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.op = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(ch, ch, kernel_size=3, padding=1),
        )

    def forward(self, x):
        return self.op(x)


class ConditionalUNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, base_ch=128, channel_mults=(1, 2, 2, 4), num_classes=10, dropout=0.1, attn_resolutions=(16,)):
        super().__init__()
        emb_dim = base_ch * 4
        self.time_mlp = nn.Sequential(
            nn.Linear(base_ch, emb_dim),
            SiLU(),
            nn.Linear(emb_dim, emb_dim),
        )
        self.label_emb = nn.Embedding(num_classes + 1, emb_dim)  # +1 for null class in CFG

        self.in_conv = nn.Conv2d(in_ch, base_ch, kernel_size=3, padding=1)

        ch = base_ch
        ds = 32
        self.down_blocks = nn.ModuleList()
        skip_channels = [ch]
        for mult in channel_mults:
            outc = base_ch * mult
            block1 = ResBlock(ch, outc, emb_dim, dropout)
            block2 = ResBlock(outc, outc, emb_dim, dropout)
            attn = SelfAttention2d(outc) if ds in attn_resolutions else nn.Identity()
            self.down_blocks.append(nn.ModuleList([block1, block2, attn]))
            ch = outc
            skip_channels.append(ch)
            if mult != channel_mults[-1]:
                self.down_blocks.append(nn.ModuleList([Downsample(ch)]))
                ds //= 2
                skip_channels.append(ch)

        self.mid = nn.ModuleList([
            ResBlock(ch, ch, emb_dim, dropout),
            SelfAttention2d(ch),
            ResBlock(ch, ch, emb_dim, dropout),
        ])

        self.up_blocks = nn.ModuleList()
        for i, mult in list(enumerate(channel_mults))[::-1]:
            outc = base_ch * mult
            self.up_blocks.append(nn.ModuleList([
                ResBlock(ch + skip_channels.pop(), outc, emb_dim, dropout),
                ResBlock(outc + skip_channels.pop(), outc, emb_dim, dropout),
                SelfAttention2d(outc) if ds in attn_resolutions else nn.Identity(),
            ]))
            ch = outc
            if i != 0:
                self.up_blocks.append(nn.ModuleList([Upsample(ch)]))
                ds *= 2

        self.out = nn.Sequential(
            nn.GroupNorm(8, ch),
            SiLU(),
            nn.Conv2d(ch, out_ch, kernel_size=3, padding=1),
        )

        self.null_label = num_classes

    def forward(self, x, t, y):
        t_emb = timestep_embedding(t, self.time_mlp[0].in_features)
        emb = self.time_mlp(t_emb) + self.label_emb(y)

        h = self.in_conv(x)
        hs = [h]
        for block in self.down_blocks:
            if len(block) == 1:
                h = block[0](h)
            else:
                h = block[0](h, emb)
                h = block[1](h, emb)
                h = block[2](h)
            hs.append(h)

        h = self.mid[0](h, emb)
        h = self.mid[1](h)
        h = self.mid[2](h, emb)

        for block in self.up_blocks:
            if len(block) == 1:
                h = block[0](h)
            else:
                h = torch.cat([h, hs.pop()], dim=1)
                h = block[0](h, emb)
                h = torch.cat([h, hs.pop()], dim=1)
                h = block[1](h, emb)
                h = block[2](h)

        return self.out(h)

In [ ]:
class GaussianDiffusion(nn.Module):
    def __init__(self, model, timesteps=1000, beta_start=1e-4, beta_end=0.02):
        super().__init__()
        self.model = model
        self.timesteps = timesteps

        # Cosine schedule is typically more stable for CIFAR-scale DDPM
        betas = cosine_beta_schedule(timesteps)
        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)

        self.register_buffer('betas', betas)
        self.register_buffer('alphas', alphas)
        self.register_buffer('alphas_cumprod', alphas_cumprod)
        self.register_buffer('alphas_cumprod_prev', alphas_cumprod_prev)
        self.register_buffer('sqrt_alphas_cumprod', torch.sqrt(alphas_cumprod))
        self.register_buffer('sqrt_one_minus_alphas_cumprod', torch.sqrt(1.0 - alphas_cumprod))
        self.register_buffer('sqrt_recip_alphas_cumprod', torch.sqrt(1.0 / alphas_cumprod))
        self.register_buffer('sqrt_recipm1_alphas_cumprod', torch.sqrt(1.0 / alphas_cumprod - 1.0))

        posterior_var = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)
        self.register_buffer('posterior_variance', posterior_var.clamp(min=1e-20))
        self.register_buffer(
            'posterior_mean_coef1',
            betas * torch.sqrt(alphas_cumprod_prev) / (1.0 - alphas_cumprod),
        )
        self.register_buffer(
            'posterior_mean_coef2',
            (1.0 - alphas_cumprod_prev) * torch.sqrt(alphas) / (1.0 - alphas_cumprod),
        )

    def q_sample(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_ac = extract(self.sqrt_alphas_cumprod, t, x0.shape)
        sqrt_om = extract(self.sqrt_one_minus_alphas_cumprod, t, x0.shape)
        return sqrt_ac * x0 + sqrt_om * noise

    def p_losses(self, x0, t, y):
        noise = torch.randn_like(x0)
        xt = self.q_sample(x0, t, noise)
        pred_noise = self.model(xt, t, y)
        return F.mse_loss(pred_noise, noise)

    def predict_x0_from_eps(self, x_t, t, eps):
        return extract(self.sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t - extract(
            self.sqrt_recipm1_alphas_cumprod, t, x_t.shape
        ) * eps

    @torch.no_grad()
    def p_sample(self, x, t, y, guidance_scale=3.0, clip_denoised=True):
        b = x.shape[0]
        t_batch = torch.full((b,), t, device=x.device, dtype=torch.long)

        # classifier-free guidance: eps = eps_uncond + s * (eps_cond - eps_uncond)
        null_y = torch.full_like(y, self.model.null_label)
        eps_uncond = self.model(x, t_batch, null_y)
        eps_cond = self.model(x, t_batch, y)
        eps = eps_uncond + guidance_scale * (eps_cond - eps_uncond)

        x0_pred = self.predict_x0_from_eps(x, t_batch, eps)
        if clip_denoised:
            x0_pred = x0_pred.clamp(-1.0, 1.0)

        model_mean = extract(self.posterior_mean_coef1, t_batch, x.shape) * x0_pred + extract(
            self.posterior_mean_coef2, t_batch, x.shape
        ) * x

        if t == 0:
            return model_mean

        noise = torch.randn_like(x)
        var = extract(self.posterior_variance, t_batch, x.shape)
        return model_mean + torch.sqrt(var) * noise

    @torch.no_grad()
    def sample(self, num_samples, image_size, y, guidance_scale=3.0, clip_denoised=True):
        x = torch.randn(num_samples, 3, image_size, image_size, device=y.device)
        for t in tqdm(reversed(range(self.timesteps)), total=self.timesteps, desc='Sampling', leave=False):
            x = self.p_sample(x, t, y, guidance_scale=guidance_scale, clip_denoised=clip_denoised)
        return x

In [ ]:
class EMA:
    def __init__(self, model, decay=0.9999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        for k, v in msd.items():
            self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1.0 - self.decay)

    def copy_to(self, model):
        model.load_state_dict(self.shadow, strict=True)


def maybe_drop_labels(y, drop_prob, null_label):
    if drop_prob <= 0:
        return y
    mask = torch.rand_like(y.float()) < drop_prob
    y = y.clone()
    y[mask] = null_label
    return y


@torch.no_grad()
def show_samples(diffusion, epoch, classes, guidance_scale=3.0, rows=10):
    diffusion.eval()
    y = torch.arange(0, len(classes), device=device).repeat_interleave(rows)
    samples = diffusion.sample(
        num_samples=len(y),
        image_size=cfg.image_size,
        y=y,
        guidance_scale=guidance_scale,
        clip_denoised=True,
    )
    samples = (samples * 0.5 + 0.5).clamp(0, 1)

    grid = utils.make_grid(samples, nrow=rows, padding=0)
    plt.figure(figsize=(rows * 1.2, len(classes) * 1.2))
    plt.imshow(np.transpose(grid.cpu().numpy(), (1, 2, 0)))
    plt.axis('off')
    plt.title(f'Epoch {epoch} | CFG={guidance_scale}')
    plt.show()

    out_path = os.path.join(cfg.out_dir, f'samples_epoch_{epoch:03d}.png')
    utils.save_image(samples, out_path, nrow=rows, padding=0)


def save_ckpt(path, model, ema_model, optimizer, scheduler, epoch):
    torch.save({
        'model': model.state_dict(),
        'ema_model': ema_model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'epoch': epoch,
        'cfg': cfg.__dict__,
    }, path)

In [ ]:
unet = ConditionalUNet(
    in_ch=3,
    out_ch=3,
    base_ch=cfg.base_channels,
    channel_mults=cfg.channel_mults,
    num_classes=cfg.num_classes,
    dropout=cfg.dropout,
    attn_resolutions=cfg.attn_resolutions,
).to(device)

diffusion = GaussianDiffusion(
    model=unet,
    timesteps=cfg.timesteps,
    beta_start=cfg.beta_start,
    beta_end=cfg.beta_end,
).to(device)

ema_model = copy.deepcopy(unet).eval().to(device)
for p in ema_model.parameters():
    p.requires_grad_(False)
ema = EMA(unet, decay=cfg.ema_decay)
ema.copy_to(ema_model)

optimizer = torch.optim.AdamW(unet.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
total_steps = cfg.epochs * len(train_loader)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_amp and device.type == 'cuda')

print(f'Parameters: {sum(p.numel() for p in unet.parameters())/1e6:.2f}M')

In [ ]:
train_losses = []

for epoch in range(1, cfg.epochs + 1):
    unet.train()
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{cfg.epochs}', leave=False)

    epoch_loss = 0.0
    for x, y in pbar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        t = torch.randint(0, cfg.timesteps, (x.shape[0],), device=device).long()
        y_dropped = maybe_drop_labels(y, cfg.label_drop_prob, unet.null_label)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=cfg.use_amp and device.type == 'cuda'):
            loss = diffusion.p_losses(x, t, y_dropped)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(unet.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        ema.update(unet)

        epoch_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}', lr=f'{optimizer.param_groups[0]["lr"]:.2e}')

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f'Epoch {epoch:03d}: loss={avg_loss:.5f}')

    if epoch % cfg.sample_every == 0 or epoch == 1:
        ema.copy_to(ema_model)
        diffusion_ema = GaussianDiffusion(
            model=ema_model,
            timesteps=cfg.timesteps,
            beta_start=cfg.beta_start,
            beta_end=cfg.beta_end,
        ).to(device)
        # lower guidance for intermediate previews avoids oversaturation/noisy artifacts
        show_samples(diffusion_ema, epoch, classes, guidance_scale=cfg.preview_guidance_scale, rows=cfg.num_sample_rows)

        ckpt_path = os.path.join(cfg.out_dir, f'diffusion_epoch_{epoch:03d}.pt')
        save_ckpt(ckpt_path, unet, ema_model, optimizer, scheduler, epoch)

plt.figure(figsize=(8, 4))
plt.plot(train_losses)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE noise prediction loss')
plt.grid(True)
plt.show()

In [ ]:
ema.copy_to(ema_model)
diffusion_ema = GaussianDiffusion(
    model=ema_model,
    timesteps=cfg.timesteps,
    beta_start=cfg.beta_start,
    beta_end=cfg.beta_end,
).to(device)

rows = 12
y = torch.arange(cfg.num_classes, device=device).repeat_interleave(rows)
with torch.no_grad():
    samples = diffusion_ema.sample(
        num_samples=len(y),
        image_size=cfg.image_size,
        y=y,
        guidance_scale=cfg.guidance_scale,
        clip_denoised=True,
    )

samples = (samples * 0.5 + 0.5).clamp(0, 1)
out_path = os.path.join(cfg.out_dir, 'final_samples.png')
utils.save_image(samples, out_path, nrow=rows, padding=0)

plt.figure(figsize=(rows * 1.1, cfg.num_classes * 1.1))
plt.imshow(np.transpose(utils.make_grid(samples, nrow=rows, padding=0).cpu().numpy(), (1, 2, 0)))
plt.axis('off')
plt.title(f'Final CIFAR-10 samples | CFG={cfg.guidance_scale}')
plt.show()

print('Saved:', out_path)